In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import os
import shutil
from marmopose.config import Config
from marmopose.utils.data_io import load_points_3d_h5


In [ ]:
config_path = '../configs/default.yaml'

config = Config(
    config_path=config_path,
    
    n_tracks=1,
    project='../demos/single',
)
idx_spinemid = config.animal['bodyparts'].index('spinemid')
idx_tailbase = config.animal['bodyparts'].index('tailbase')
idx_neck = config.animal['bodyparts'].index('neck')


In [ ]:
video = '260703/Home'
video_path = f"/srv/MarmOT/VideoTracking/Videos/{video}/Input_preprocessed"
ncams = 4 if video[-4:] == 'Etho' else 6
# points_3d = load_points_3d_h5(f"/scratch/VideoTracking/Videos/{video}/Output_basemodel/points_3d/optimized.h5")
f = 1513
jitter = 10
duration = 20
step=1
marm = True

In [ ]:
%matplotlib inline
plt.clf()
shutil.rmtree('tmp') 
os.mkdir('tmp')
sizefig= (30,20) if ncams == 4 or marm else (60,40)
# fig, ax = plt.subplots(figsize=(16, 4))
# ax.axis('off')
# norm = mcolors.Normalize(vmin=-20, vmax=20)
# data = points_3d[0,f + 1,:,:] - points_3d[0,f,:,:]
# data = np.concatenate((data,np.mean(data,axis=0,keepdims=True)),axis=0)
# data = np.concatenate((data,np.sqrt(np.einsum("ij,ij->i", data, data))[:,np.newaxis]),axis=1).transpose((1,0))
# print(data.shape)
# cmap = plt.cm.RdYlBu
# table = ax.table(
#     cellText=np.round(data,decimals=2),
#     cellColours= cmap(norm(data)),
#     rowLabels=['Velocity X','Velocity Y','Velocity Z','Velocity magnitude'],
#     colLabels=config.animal['bodyparts'] + ['Mean'],
#     loc='center',
#     cellLoc='center',
#     fontsize = 20
# )
# plt.show()
for d in range(-jitter * step,(duration - jitter) * step, step):
    fig, axs = plt.subplots(ncams//2,2,figsize=sizefig)
    axs = axs.flatten()
    fig_thresh, axs_thresh = plt.subplots(ncams//2,2,figsize=(30,20))
    axs_thresh = axs_thresh.flatten()
    for i in range(1,ncams + 1):
        vidcap = cv2.VideoCapture(os.path.join(video_path,f'output{i}.mp4'))
        vidcap.set(cv2.CAP_PROP_POS_FRAMES, f + d)
        success, frame = vidcap.read()
        if not success:
            print(f"Couldn't read frame {f + d} in video {os.path.join(video_path,f'output{i}.mp4')}")
            continue
        red = frame[..., 2]
        # clahe = cv2.createCLAHE(clipLimit=0.3, tileGridSize=(8,8))
        # red = clahe.apply(red)
        max_red = np.max(red)
        thresholded_frame = np.logical_and(cv2.threshold(red, max_red * 0.9, max_red, cv2.THRESH_BINARY)[1],~np.logical_or(cv2.threshold(frame[..., 0], max_red * 0.75, max_red, cv2.THRESH_BINARY)[1],cv2.threshold(frame[..., 1], max_red * 0.75, max_red, cv2.THRESH_BINARY)[1]))
        axs[i-1].imshow(frame[..., ::-1])
        axs[i-1].axis('off')
        axs[i-1].set_title(f'Frame {f + d}')
        # thresholded_frame = cv2.GaussianBlur(thresholded_frame.astype(float), (5,5), 0)
        # thresholded_frame[thresholded_frame < 0.5] = 0
        # thresholded_frame[thresholded_frame >= 0.5] = 1
        if not marm:
            axs_thresh[i-1].imshow(thresholded_frame,interpolation='none')
            print(i, d, np.sum(thresholded_frame))
            axs_thresh[i-1].axis('off')
            axs_thresh[i-1].set_title(f'Frame {f + d}')
    fig.tight_layout()
    fig.savefig(f'tmp/{d}.png')
    if not marm:
        fig_thresh.tight_layout()
        fig_thresh.savefig(f'tmp/thresh{d}.png')
    plt.cla()
    plt.close(fig)
    plt.close(fig_thresh)



In [ ]:

%matplotlib widget
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
for bodyparts in config.visualization['skeleton'][::-1]:
    idx_bodyparts = []
    for bodypart in bodyparts:
        idx_bodyparts.append(config.animal['bodyparts'].index(bodypart))
    ax.plot(points_3d[0,f,idx_bodyparts,0],points_3d[0,f,idx_bodyparts,1],points_3d[0,f,idx_bodyparts,2], marker = 'o', ms=3,c='b')
    ax.plot(points_3d[0,f+1,idx_bodyparts,0],points_3d[0,f+1,idx_bodyparts,1],points_3d[0,f+1,idx_bodyparts,2], marker = 'o', ms=3,c='r')
fig.show()
